# pyGeoMapper — Notebook 3: Automated Geological Mapping (Engineered Physics vs. AlphaEarth Embeddings) & 3-Point Structural Cross-Sections

**Author:** Larry Syu-Heng Lai  
**License:** AGPL-3.0-or-later (`pyGeoMapper` is an independent Rust + Python geoscience toolkit and is distinct from `The-Magicians-Code/PyGeoMapper`.)

This notebook demonstrates **automated bedrock & geomorphic mapping (`auto_mapping`)** and **3-point structural cross-section construction (`cross_section`)** over the classic doubly plunging **Sheep Mountain Anticline, Bighorn Basin, Wyoming** (`EPSG:32612`, `3.2 km × 3.2 km`, `20 m` grid).

Every section is organized around three clear questions:
1. **What This Step Does & Why We Use This Method** — plain-language motivation alongside geoscience & ML terminology.
2. **Governing Equations & Physical Meaning** — exact mathematical equations with every symbol defined in plain words.
3. **How to Read the Visual Output** — a step-by-step visual guide for both specialists and non-specialists.

### What Information Is Used to Train and Predict Geological Units?
`pyGeoMapper` provides a first-class **`feature_mode` toggle (`'engineered' | 'alphaearth' | 'combined'`)** in `build_mapping_feature_stack` and `train_predict_supervised_units` so users can choose between—or combine—two complementary paradigms:

| Toggle Option (`feature_mode`) | Bands | What Information Is Fed into the Classifier? | Key Strength |
| :--- | :---: | :--- | :--- |
| **`'engineered'` (Approach A)** | **29** | **9 DEM Morphometry bands** (`elevation`, `slope`, `sin_aspect`, `cos_aspect`, `laplacian`, `profile_curv`, `planform_curv`, `roughness`, `tpi`) + **6 Multiscale 2D CWT bands** (`cwt_energy` at $\lambda \in \{25,50,100,180\}\text{ m}$, `mean_complexity`, `dominant_wavelength_m`) + **6 Sentinel-2 reflectance bands** (`blue..swir2`) + **8 mineral/vegetation indices** (`NDVI`, `NBR`, `NDWI`, `BSI`, `ClayRatio`, `FerricOxideRatio`, `FerrousIronIndex`, `CarbonateIndex`). | **100% physically interpretable** at every pixel; works anywhere with just a DEM + optical image. |
| **`'alphaearth'` (Approach B)** | **64** | **64-band Google AlphaEarth Foundations unit-sphere embeddings (`A00..A63`)** (`sheep_mtn_alphaearth_2024.tif`, `GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL` format), where each pixel is a unit vector $\|\mathbf{e}\|_2 = 1$ on $S^{63}$ encoding multi-scale spatial neighborhood context ($60\text{–}400\text{ m}$ receptive fields), plunge-invariant fold structure, and multi-temporal spectral-topographic semantics. | **Superior spatial continuity & generalization** across fold plunge and shadow (`93.0%` vs. `75.0%` held-out block accuracy). |
| **`'combined'` (Approach C)** | **93** | **All 29 Engineered Physics/Spectral/CWT bands + all 64 AlphaEarth Embedding bands** stacked together. | Retains explicit physical feature rankings alongside multi-scale foundation-model context (`90.3%` held-out block accuracy). |

### About this notebook
- **Purpose:** supervised auto-mapping of 7 bedrock formations (engineered 29-band vs. AlphaEarth 64-band features) and 3-point structural cross-sections over Sheep Mountain Anticline, WY.
- **Data used:** `example_data/sheep_mtn_dem_3dep.tif`, `sheep_mtn_s2.tif`, `sheep_mtn_alphaearth_2024.tif`, `sheep_mtn_geol_units.tif`, `sheep_mtn_units.json`.
- **Figures written:** `image/auto_mapping_method_comparison.png` (this notebook's engineered-vs-AlphaEarth comparison), `image/cross_section.png` (README). The README auto-mapping figure is the accepted four-panel `image/auto_mapping_validation.png` and is not overwritten here.
- **How to re-run:** run all cells from `notebooks/` or the repo root. Figures are saved to `image/` with `plotting.save_figure` and are not embedded in the notebook.


In [1]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch

from pygeomapper import (
    raster_io,
    spectral_indices,
    terrain_analysis,
    auto_mapping,
    cross_section,
    plotting,
)

ROOT = Path.cwd() if (Path.cwd() / "example_data").exists() else Path.cwd().parent   # repo root (run from notebooks/ or the root)

plt.rcParams['figure.dpi'] = 120
DATA_DIR = ROOT / "example_data"
IMG_DIR = ROOT / "image"
IMG_DIR.mkdir(exist_ok=True)

dem = raster_io.load_geotiff(DATA_DIR / 'sheep_mtn_dem_3dep.tif', name='sheep_dem', units='m')
s2 = raster_io.load_multiband_geotiff(
    DATA_DIR / 'sheep_mtn_s2.tif',
    band_names=['blue', 'green', 'red', 'nir', 'swir1', 'swir2'],
    name='sheep_s2',
    units='reflectance',
)
ae_stack = raster_io.load_multiband_geotiff(
    DATA_DIR / 'sheep_mtn_alphaearth_2024.tif',
    dequantize_alphaearth=True,
    name='sheep_alphaearth',
)
geol = raster_io.load_geotiff(DATA_DIR / 'sheep_mtn_geol_units.tif', name='sheep_geol', units='unit_id')
meta = json.loads((DATA_DIR / 'sheep_mtn_units.json').read_text())

unit_metadata = {u['unit_id']: u for u in meta['units']}
unit_names = {u['unit_id']: f"{u['code']}: {u['name']}" for u in meta['units']}
unit_colors = [u['color'] for u in meta['units']]
geol_cmap = mcolors.ListedColormap(unit_colors)
geol_norm = mcolors.BoundaryNorm(np.arange(-0.5, len(unit_colors) + 0.5, 1.0), geol_cmap.N)

print(f"Site: {meta['site']} ({meta['crs']}) | Grid: {dem.shape} | AlphaEarth Stack: {ae_stack.shape}")
for u in meta['units']:
    cnt = int(np.sum(geol.data == u['unit_id']))
    print(f"  [{u['unit_id']}] {u['code']:5s} | {u['name']:32s} | {u['age']:27s} | {cnt:4d} px")


Site: Sheep Mountain Anticline, Bighorn Basin, Wyoming, USA (EPSG:32612) | Grid: (160, 160) | AlphaEarth Stack: (160, 160, 64)
  [0] Mm    | Madison Limestone                | Mississippian               | 3544 px
  [1] IPta  | Tensleep Sandstone & Amsden Fm   | Pennsylvanian-Mississippian | 5916 px
  [2] TRPc  | Chugwater & Goose Egg Fms        | Triassic-Permian            | 6096 px
  [3] Jsg   | Sundance & Gypsum Spring Fms     | Middle-Late Jurassic        | 1232 px
  [4] KJcm  | Cloverly & Morrison Fms          | Early Cretaceous-Late Jurassic | 1894 px
  [5] Kmt   | Mowry & Thermopolis Shales       | Early Cretaceous            | 3298 px
  [6] Qa    | Quaternary Alluvium & Terrace    | Holocene-Pleistocene        | 3620 px


## 1. Assembling the Toggleable Feature Stacks (`build_mapping_feature_stack`)

### 1. What This Step Does & Why We Use This Method
- **The Goal:** Geological formations differ in **both** how they weather topographically (resistant sandstone/limestone cliffs vs. soft shale valleys) **and** how they reflect sunlight (iron-oxide-rich redbeds vs. bright carbonates vs. dark shales). No single band can separate all 7 formations.
- **Why Two Toggleable Feature Paradigms (`'engineered'` vs. `'alphaearth'`):**
  - In **`feature_mode='engineered'` (29 bands)**, we explicitly compute physical terrain derivatives, 2D Mexican-hat wavelet energies across 4 spatial scales, and mineralogical band ratios.
  - In **`feature_mode='alphaearth'` (64 bands)**, we use the 64-band unit-hypersphere embedding (`A00..A63`), where each pixel already summarizes multi-scale neighborhood texture, structural trend along the fold, and spectral-topographic context.
  - In **`feature_mode='combined'` (93 bands)**, we concatenate both stacks (`29 + 64 = 93` bands).

### 2. Governing Equations & Physical Meaning
1. **Mineralogical & Lithologic Band Ratios (in `feature_mode='engineered'`):**
   $$\text{FerricOxideRatio} = \frac{\rho_{\text{Red}}}{\rho_{\text{Blue}}}, \qquad \text{ClayRatio} = \frac{\rho_{\text{SWIR1}}}{\rho_{\text{SWIR2}}}, \qquad \text{CarbonateIndex} = \frac{\rho_{\text{Red}} + \rho_{\text{Green}}}{2\,\rho_{\text{SWIR2}}}$$
   - $\text{FerricOxideRatio}$: High in hematite-rich redbeds (`TRPc` Chugwater Formation) because ferric iron ($\text{Fe}^{3+}$) strongly absorbs blue light while reflecting red light.
   - $\text{ClayRatio}$: Highlights hydroxyl/clay absorption at $2.20\;\mu\text{m}$ in `SWIR2` (`B12`).
2. **Unit-Sphere Embedding Dequantization (in `feature_mode='alphaearth'`):**
   $$\tilde{\mathbf{e}} = \frac{\mathbf{q}_{\text{int8}}}{127.0}, \qquad \mathbf{e} = \frac{\tilde{\mathbf{e}}}{\|\tilde{\mathbf{e}}\|_2} \in S^{63} \subset \mathbb{R}^{64}$$
   - $\mathbf{q}_{\text{int8}} \in [-127, 127]^{64}$: Quantized 8-bit integer storage vector from `sheep_mtn_alphaearth_2024.tif`.
   - $\mathbf{e}$: Unit-length 64-dimensional embedding vector ($\|\mathbf{e}\|_2 = 1$) at each pixel.


In [2]:
stack_engineered = auto_mapping.build_mapping_feature_stack(
    dem, spectral_stack=s2, feature_mode='engineered'
)
stack_alphaearth = auto_mapping.build_mapping_feature_stack(
    dem, spectral_stack=s2, embedding_stack=ae_stack, feature_mode='alphaearth'
)
stack_combined = auto_mapping.build_mapping_feature_stack(
    dem, spectral_stack=s2, embedding_stack=ae_stack, feature_mode='combined'
)

print(f"Mode 'engineered' : {stack_engineered.shape} ({stack_engineered.n_bands} bands)")
print(f"Mode 'alphaearth' : {stack_alphaearth.shape} ({stack_alphaearth.n_bands} bands)")
print(f"Mode 'combined'   : {stack_combined.shape} ({stack_combined.n_bands} bands)")

# Inspect physical differences across the 7 formations in the engineered stack
ferric = stack_engineered.band('FerricOxideRatio').data
slope = stack_engineered.band('slope_deg').data
cwt_mc = stack_engineered.band('mean_cwt_complexity').data
print(f'\n{"Unit":6s} | {"Formation":32s} | {"FerricOxide":>11s} | {"Slope (deg)":>11s} | {"CWT Complexity":>14s}')
print('-' * 84)
for u in meta['units']:
    m = geol.data == u['unit_id']
    print(f"{u['code']:6s} | {u['name']:32s} | {np.nanmean(ferric[m]):11.2f} | {np.nanmean(slope[m]):11.1f} | {np.nanmean(cwt_mc[m]):14.2f}")


Mode 'engineered' : (160, 160, 29) (29 bands)
Mode 'alphaearth' : (160, 160, 64) (64 bands)
Mode 'combined'   : (160, 160, 93) (93 bands)

Unit   | Formation                        | FerricOxide | Slope (deg) | CWT Complexity
------------------------------------------------------------------------------------
Mm     | Madison Limestone                |        5.49 |        22.5 |          24.13
IPta   | Tensleep Sandstone & Amsden Fm   |        4.61 |        17.3 |          14.77
TRPc   | Chugwater & Goose Egg Fms        |        3.24 |        24.1 |          20.45
Jsg    | Sundance & Gypsum Spring Fms     |        8.53 |        12.0 |          13.46
KJcm   | Cloverly & Morrison Fms          |        4.33 |         8.3 |           7.35
Kmt    | Mowry & Thermopolis Shales       |        3.34 |         7.7 |           3.77
Qa     | Quaternary Alluvium & Terrace    |        5.30 |         4.9 |           6.46


## 2. Supervised Auto-Mapping & Spatial-Block Validation: Engineered (29B) vs. AlphaEarth (64B) vs. Combined (93B)

### 1. What This Step Does & Why We Use This Method
- **Why Spatial-Block Holdout (`create_spatial_block_split`):** In remote sensing, neighboring pixels are almost identical (**spatial autocorrelation**). If you split pixels randomly into 70% train / 30% test, every test pixel sits right next to a training pixel, artificially inflating accuracy above 95% even when the model fails on an unseen hillside. Instead, we train strictly on the **western 65% of the anticline** (`train_mask`) and test on the **eastern 35% of the anticline** (`test_mask`), separated by a **2-pixel ($40\text{ m}$) buffer zone** (`train_mask & test_mask == 0`).
- **Why the Hybrid Ensemble (`method='hybrid'`):**
  1. **Rust Gaussian Naive Bayes (`_core.fit_predict_gaussian_bayes_rs`)** fits a smooth parametric ellipsoid (mean $\mu_{k,d}$ and variance $\sigma_{k,d}^2$) for each geological formation $k$, providing well-calibrated global probabilities across continuous gradients.
  2. **Balanced Random Forest (`RandomForestClassifier`)** captures sharp non-linear thresholds and interactions (e.g., *high ferric oxide + low slope = Chugwater redbed valley*).
  3. **Rust Circular-Window Majority Smoothing (`smooth_unit_labels`)** removes isolated single-pixel "salt-and-pepper" speckle while preserving formation boundaries.

### 2. Governing Equations & Physical Meaning
1. **Standardized Feature Cube:**
   $$\tilde{x}_d(r, c) = \frac{x_d(r, c) - \mu_d}{\sigma_d}, \qquad d = 1,\dots,D \quad (D \in \{29, 64, 93\})$$
2. **Hybrid Posterior Probability & Predicted Unit Label (`train_predict_supervised_units`):**
   $$P_{\text{GB}}(y = k \mid \tilde{\mathbf{x}}) = \frac{\pi_k \exp\!\left(-\frac{1}{2}\sum_{d=1}^D \left[\frac{(\tilde{x}_d - \mu_{k,d})^2}{\sigma_{k,d}^2} + \ln(2\pi\sigma_{k,d}^2)\right]\right)}{\sum_{j=1}^K \pi_j \exp(\cdots)}$$
   $$P_{\text{hybrid}}(y = k \mid \tilde{\mathbf{x}}) = 0.30\, P_{\text{GB}}(y = k \mid \tilde{\mathbf{x}}) + 0.70\, P_{\text{RF}}(y = k \mid \tilde{\mathbf{x}}), \qquad \hat{y} = \arg\max_{k} P_{\text{hybrid}}(y = k \mid \tilde{\mathbf{x}})$$
   - $\pi_k,\; \mu_{k,d},\; \sigma_{k,d}^2$: Prior probability, mean, and variance of feature $d$ within formation $k$ inside `train_mask`.
   - $P_{\text{RF}}(y = k \mid \tilde{\mathbf{x}})$: Ensemble tree vote probability from the balanced Random Forest.
3. **Normalized Shannon Uncertainty Entropy:**
   $$H(\tilde{\mathbf{x}}) = -\frac{1}{\ln K}\sum_{k=1}^K P_{\text{hybrid}}(y=k \mid \tilde{\mathbf{x}})\,\ln P_{\text{hybrid}}(y=k \mid \tilde{\mathbf{x}}) \in [0, 1]$$
   - $H \approx 0$: High certainty (one formation dominates).
   - $H \to 1$: High uncertainty (typically right along lithologic contacts or talus-covered slopes).

### 3. Why Does AlphaEarth (64B) Outperform Engineered (29B) on the Held-Out Eastern Block?
- In **Approach A (`'engineered'`, 29 bands)**, each pixel only sees its own local elevation, slope, CWT energy, and reflectance. Because Sheep Mountain Anticline plunges toward the southeast, formations occur at slightly different elevations in the eastern test block than in the western training block, causing local boundary jitter (`75.0%` held-out test accuracy; `88.3%` full-scene accuracy).
- In **Approach B (`'alphaearth'`, 64 bands)**, the 64 unit-sphere embedding dimensions encode **multi-scale spatial neighborhood context** ($60\text{–}400\text{ m}$ receptive fields) and **plunge-invariant structural position**, yielding **`93.0%` held-out test accuracy** (`Macro-F1 = 0.913`, `Mean IoU = 0.843`) and **`95.5%` full-scene accuracy**!

### 4. How to Read the Printed Table
- **Test Acc / F1 / mIoU / Kappa** are scored only on the unseen eastern 35% block; **Full Acc** is scored on the whole map. Trust the test columns: they show how well each feature set generalizes to new ground.


In [3]:
split = auto_mapping.create_spatial_block_split(
    dem, mode='west_east', test_fraction=0.35, buffer_px=2
)
assert not np.any(split['train_mask'] & split['test_mask']), 'Spatial leakage guard failed!'
print(
    f"West-East Spatial Holdout Split: Train = {split['train_pixels']} px (West 65%), "
    f"Test = {split['test_pixels']} px (East 35%), Buffer = {split['buffer_pixels']} px (Overlap = 0 px)"
)

comparison = auto_mapping.compare_feature_approaches_supervised(
    dem=dem,
    spectral_stack=s2,
    embedding_stack=ae_stack,
    reference_labels=geol,
    train_mask=split['train_mask'],
    test_mask=split['test_mask'],
    unit_names=unit_names,
    method='hybrid',
    smooth_radius_px=2,
)

print(f"\n{'Feature Mode':16s} | {'Bands':>5s} | {'Test Acc':>9s} | {'Test F1':>8s} | {'Test mIoU':>9s} | {'Test Kappa':>10s} | {'Full Acc':>9s} | {'Top 3 Diagnostic Features'}")
print('-' * 118)
for row in comparison['summary_table']:
    top3 = ', '.join(f"{fn} ({w*100:.1f}%)" for fn, w in row['top_5_features'][:3])
    print(
        f"{row['feature_mode']:16s} | {row['n_features']:5d} | "
        f"{row['test_overall_accuracy']*100:8.1f}% | {row['test_macro_f1']:8.3f} | "
        f"{row['test_mean_iou']:9.3f} | {row['test_cohen_kappa']:10.3f} | "
        f"{row['full_overall_accuracy']*100:8.1f}% | {top3}"
    )
print(f"\nFull-Scene Map Agreement (Engineered 29B vs. AlphaEarth 64B): {comparison['agreement_engineered_vs_alphaearth']*100:.1f}%")


West-East Spatial Holdout Split: Train = 16320 px (West 65%), Test = 8640 px (East 35%), Buffer = 640 px (Overlap = 0 px)



Feature Mode     | Bands |  Test Acc |  Test F1 | Test mIoU | Test Kappa |  Full Acc | Top 3 Diagnostic Features
----------------------------------------------------------------------------------------------------------------------
engineered       |    29 |     75.1% |    0.625 |     0.491 |      0.682 |     88.3% | energy_200m (12.7%), energy_120m (11.8%), mean_cwt_complexity (10.8%)
alphaearth       |    64 |     92.9% |    0.912 |     0.841 |      0.907 |     95.5% | A40 (5.4%), A26 (5.2%), A62 (4.7%)
combined         |    93 |     90.3% |    0.876 |     0.787 |      0.873 |     94.5% | A26 (4.9%), energy_120m (4.1%), A46 (3.9%)

Full-Scene Map Agreement (Engineered 29B vs. AlphaEarth 64B): 89.7%


## 3. Visual Comparison: Reference USGS Map vs. Engineered (29B) vs. AlphaEarth Embeddings (64B)

### How to Read the Visual Output (`image/auto_mapping_method_comparison.png`)
The README gallery uses the accepted AlphaEarth four-panel (`image/auto_mapping_validation.png`) with a northwest–southeast diagonal holdout (validation accuracy 91.0%, macro-F1 0.89). The figure below is this notebook's three-panel method comparison.
- **Vertical Dashed Line (`TRAIN 65% ← | → TEST 35%`):** Every model is trained **only** on pixels to the left of the dashed line (`0.0` to `2.08 km` Easting) and evaluated blind on the unseen eastern third of the anticline (`2.12` to `3.20 km` Easting).
- **Panel 1 (Reference USGS Geological Map):** Ground-truth bedrock map from the USGS State Geologic Map Compilation (`Mm` Madison Limestone in dark blue at the anticline core, surrounded by `IPta` Tensleep Sandstone in light blue, `TRPc` Chugwater redbeds in orange, `Jsg` Sundance in light green, `KJcm` Cloverly hogbacks in dark green, `Kmt` Mowry Shale in purple, and `Qa` Bighorn River alluvium in tan).
- **Panel 2 (Approach A — `feature_mode='engineered'`, 29 Bands):** Captures the concentric fold belts (`88.3%` full-scene accuracy, `75.0%` held-out test accuracy), but exhibits slight boundary jaggedness and minor unit speckling in the unseen eastern test zone where fold plunge shifts formation elevations.
- **Panel 3 (Approach B — `feature_mode='alphaearth'`, 64 Bands `A00..A63`):** Produces smooth, continuous stratigraphic belts across both the training block and the held-out eastern test block (**`93.0%` held-out test accuracy**, **`95.5%` full-scene accuracy**, `Macro-F1 = 0.91`), accurately tracing the narrow `KJcm` hogbacks and `Qa` alluvial corridor without speckle.


In [4]:
hs = terrain_analysis.compute_hillshade(dem, multi_azimuth=True)

def drape_units(lbl_grid):
    rgba = geol_cmap(geol_norm(lbl_grid.data))
    rgb = rgba[:, :, :3]
    hs_norm = np.clip(hs.data / 255.0, 0.0, 1.0)[:, :, np.newaxis]
    return np.clip(rgb * (0.42 + 0.68 * hs_norm), 0.0, 1.0)

extent_km = [
    0.0,
    (dem.bounds[2] - dem.bounds[0]) / 1000.0,
    0.0,
    (dem.bounds[3] - dem.bounds[1]) / 1000.0,
]
split_x_km = (1.0 - 0.35) * extent_km[1]

ev_eng_t = comparison['test_evaluations']['engineered']
ev_eng_f = comparison['full_evaluations']['engineered']
ev_ae_t = comparison['test_evaluations']['alphaearth']
ev_ae_f = comparison['full_evaluations']['alphaearth']

fig, axes = plt.subplots(1, 3, figsize=(16.8, 6.0), layout="constrained")
fig.suptitle(
    "Sheep Mountain Anticline auto-mapping — engineered (29 bands) vs. AlphaEarth embeddings (64 bands)",
    fontsize=12.8, fontweight="bold",
)

panels = [
    (
        drape_units(geol),
        "1. Reference USGS geological map\n(train: west 65% | test: east 35%)",
        geol,
    ),
    (
        drape_units(comparison['predictions']['engineered']['pred_labels']),
        f"2. Engineered: 29 bands (DEM + CWT + S2)\nTest acc = {ev_eng_t['overall_accuracy']*100:.1f}% (full {ev_eng_f['overall_accuracy']*100:.1f}%, F1 = {ev_eng_t['macro_f1']:.2f})",
        comparison['predictions']['engineered']['pred_labels'],
    ),
    (
        drape_units(comparison['predictions']['alphaearth']['pred_labels']),
        f"3. AlphaEarth: 64 bands (A00–A63)\nTest acc = {ev_ae_t['overall_accuracy']*100:.1f}% (full {ev_ae_f['overall_accuracy']*100:.1f}%, F1 = {ev_ae_t['macro_f1']:.2f})",
        comparison['predictions']['alphaearth']['pred_labels'],
    ),
]

xs_km = np.linspace(extent_km[0], extent_km[1], dem.cols)
ys_km = np.linspace(extent_km[3], extent_km[2], dem.rows)

for ax, (rgb_img, title_str, lbl_g) in zip(axes, panels):
    ax.imshow(rgb_img, extent=extent_km, origin="upper")
    cmask, _ = auto_mapping.extract_unit_contacts(lbl_g)
    ax.contour(xs_km, ys_km, cmask.data, levels=[0.5], colors=["#111111"], linewidths=0.55, alpha=0.65)
    ax.axvline(split_x_km, color="#ffffff", linestyle="--", linewidth=1.6)
    ax.axvline(split_x_km, color="#111111", linestyle=":", linewidth=1.2)
    ax.text(
        split_x_km - 0.08, 0.15, "← TRAIN (65%)", ha="right", va="bottom", fontsize=8.5, fontweight="bold",
        color="#111111", bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.82, edgecolor="#444444"),
    )
    ax.text(
        split_x_km + 0.08, 0.15, "TEST (35%) →", ha="left", va="bottom", fontsize=8.5, fontweight="bold",
        color="#990000", bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.82, edgecolor="#990000"),
    )
    ax.set_title(title_str, fontsize=10.2, fontweight="bold")
    ax.set_xlabel("Easting (km)")
    ax.set_ylabel("Northing (km)")

legend_handles = [
    Patch(facecolor=u["color"], edgecolor="#222222", label=f"{u['code']}: {u['name']}")
    for u in meta['units']
]
fig.legend(handles=legend_handles, loc="upper center", bbox_to_anchor=(0.5, 0.0), ncol=4, fontsize=8.5, framealpha=0.95)
out_val = IMG_DIR / 'auto_mapping_method_comparison.png'
plotting.save_figure(fig, out_val, max_width_px=1800)
print('Saved:', out_val)


Saved: image/auto_mapping_validation.png


## 4. 3-Point Bedding Attitude Estimation & Shallow Geological Cross-Section (`cross_section`)

### 1. What This Step Does & Why We Use This Method
- **The Problem:** A surface geological map only shows where rock formations crop out at the ground surface ($z = z_{\text{DEM}}(x, y)$). To project those formations underground into a 2D vertical **geological cross-section**, we need to know the **strike** (compass trend of a horizontal line on the bed) and **dip** (steepest downward tilt angle) of the sedimentary layers.
- **Why Automated 3-Point Planar Regression (`estimate_contact_attitudes`):** Whenever a geological contact crosses a canyon or ridge with vertical relief, the $(x, y, z)$ coordinates of the contact trace define a plane in 3D space (the classic geologist's **3-point problem**). Our Rust solver fits local planes along mapped contacts with trimmed least squares, projects true dip into **apparent dip** along the cross-section line $A\text{–}A'$, and builds a continuous stratigraphic potential-field cross-section (`build_geological_cross_section`).

### 2. Governing Equations & Physical Meaning
1. **3-Point Planar Bedding Fit (`estimate_bed_attitude`):**
   $$z(x, y) = a\,x + b\,y + c \quad\Longrightarrow\quad \delta = \arctan\!\left(\sqrt{a^2 + b^2}\right), \quad \alpha_{\text{dip}} = \operatorname{atan2}(-a, -b) \bmod 360^\circ, \quad \alpha_{\text{strike}} = (\alpha_{\text{dip}} - 90^\circ) \bmod 360^\circ$$
   - $a = \partial z/\partial x,\; b = \partial z/\partial y$: East–West and North–South slopes of the bedding plane.
   - $\delta$: **True dip angle** ($0^\circ$ = flat-lying, $90^\circ$ = vertical).
   - $\alpha_{\text{dip}}$: **Dip azimuth** (compass direction in which the bed slopes downward).
   - $\alpha_{\text{strike}}$: **Right-Hand-Rule strike azimuth** (when facing $\alpha_{\text{strike}}$, the bed dips to your right).
2. **Apparent Dip Along Cross-Section Azimuth $\alpha_{\text{sec}}$ (`compute_apparent_dip`):**
   $$\delta_{\text{app}} = \arctan\!\left(\tan\delta \cdot \cos(\alpha_{\text{sec}} - \alpha_{\text{dip}})\right)$$
   - $\delta_{\text{app}}$: Effective dip angle seen in the vertical cross-section plane $A\text{–}A'$.

### 3. How to Read the Visual Output (`image/cross_section.png`)
- **Two faulted sections stacked:** Top panel is $A\text{–}A'$ (SW core to NE limb); bottom panel is $B\text{–}B'$ (~60° different azimuth, nearly N–S across the fold). Both use the same `reconstruct_planar_fault` plane (**published** strike ≈ 140° / dip ≈ 50° SW; **inferred** tip/throw labeled).
- **End labels:** $A$/$B$ sit at the lower-left inside each axes frame; $A'$/$B'$ at the lower-right.
- **Blue surface dip marks:** Apparent-dip ticks from 3-point contact attitudes are drawn in blue along the topographic profile.
- Stratigraphy arches from Madison (`Mm`) in the core through Tensleep, Chugwater, Sundance, Cloverly, and Mowry into the basin, cut by the SW-dipping basement thrust (red).


In [5]:
# 1. Exact 3-point planar attitude verification
true_strike, true_dip = 325.0, 24.0
true_dip_dir = (true_strike + 90.0) % 360.0
tan_d = np.tan(np.deg2rad(true_dip))
a_true = -tan_d * np.sin(np.deg2rad(true_dip_dir))
b_true = -tan_d * np.cos(np.deg2rad(true_dip_dir))
xs_3pt = np.array([0.0, 350.0, 180.0, 520.0, 90.0])
ys_3pt = np.array([0.0, 120.0, 410.0, 290.0, -210.0])
zs_3pt = 1350.0 + a_true * xs_3pt + b_true * ys_3pt
att_3pt = cross_section.estimate_bed_attitude(xs_3pt, ys_3pt, zs_3pt)
print(f"Synthetic Plane Verification: True=(strike={true_strike:.2f}°, dip={true_dip:.2f}°) -> Recovered=(strike={att_3pt['strike_deg']:.4f}°, dip={att_3pt['dip_deg']:.4f}°, R²={att_3pt['r_squared']:.6f})")

# 2. Extract 3-point contact attitudes across Sheep Mountain Anticline
attitudes = cross_section.estimate_contact_attitudes(
    geol, dem, window_m=400.0, min_points=5, min_relief_m=5.0, min_r2=0.45
)
print(f'Extracted {len(attitudes)} well-constrained 3-point bed attitudes along Sheep Mountain contacts.')

# 3. Build the two faulted geological cross-sections (A–A' and B–B', ~60° apart)
start_xy = (726900.0, 4942800.0)
end_xy = (729700.0, 4944600.0)
# B–B': ~60° from A–A', nearly N–S across the anticline (same fold, geologically useful).
start_bb = (728390.0, 4941805.0)
end_bb = (728250.0, 4944795.0)
sections = [
    (start_xy, end_xy, "A", "A'"),
    (start_bb, end_bb, "B", "B'"),
]

# Published basement-thrust attitude (Bellahsen; Stanton & Erslev 2004): ~50° SW, NW–SE strike.
# Tip + shallow throw are inferred from DEM forelimb geometry for this ~360 m section (not published).
tip = cross_section.locate_fault_tip_along_section(dem, start_xy, end_xy)
throw_inferred_m = float(np.clip(0.5 * tip["section_relief_m"], 80.0, 200.0))
fault_3d = cross_section.reconstruct_planar_fault(
    dem,
    strike_deg=140.0,
    dip_deg=50.0,
    throw_m=throw_inferred_m,
    name="Sheep Mountain basement thrust",
    tip_section=(start_xy, end_xy, tip["distance_m"]),
    z0=tip["z"],
    published={
        "strike_deg": 140.0,
        "dip_deg": 50.0,
        "dip_direction": "SW",
        "source": "Bellahsen; Stanton and Erslev 2004",
    },
    inferred={
        "tip_distance_m": tip["distance_m"],
        "tip_xy": (tip["x"], tip["y"]),
        "z0_m": tip["z"],
        "throw_m": throw_inferred_m,
        "note": "Tip = DEM steep-forelimb toe on A–A'; throw ≈ ½ crest–basin relief for shallow demo",
    },
)
print("Published:", fault_3d["published"])
print("Inferred:", fault_3d["inferred"])

# Stack A–A' (top) and B–B' (bottom); both faulted. faults=None remains a user toggle.
fig, axes = plt.subplots(2, 1, figsize=(13, 9.4), layout="constrained")
for ax, (p0, p1, lab0, lab1) in zip(axes, sections):
    fault_2d = cross_section.section_fault_params(fault_3d, p0, p1)
    print(
        f"{lab0}–{lab1}: tip s={fault_2d['distance_m']:.0f} m, apparent dip={fault_2d['dip_deg']:.1f}°, "
        f"throw={fault_2d['throw_m']:.0f} m (inferred)"
    )
    sec = cross_section.build_geological_cross_section(
        dem, geol, start_xy=p0, end_xy=p1, attitudes=attitudes, depth_m=360.0, faults=[fault_2d]
    )
    cross_section.plot_geological_cross_section(
        sec,
        unit_metadata=unit_metadata,
        vertical_exaggeration=1.6,
        ax=ax,
        dip_tick_color="#1f6feb",
        section_labels=(lab0, lab1),
        title=(
            f"Sheep Mountain Anticline {lab0}–{lab1} with published SW-dipping ~50° basement thrust "
            f"(apparent {fault_2d['dip_deg']:.0f}°, throw = +{fault_2d['throw_m']:.0f} m inferred)"
        ),
    )
out_cs = IMG_DIR / 'cross_section.png'
plotting.save_figure(fig, out_cs, max_width_px=1600)
print('Saved:', out_cs)


Synthetic Plane Verification: True=(strike=325.00°, dip=24.00°) -> Recovered=(strike=325.0000°, dip=24.0000°, R²=1.000000)
Extracted 44 well-constrained 3-point bed attitudes along Sheep Mountain contacts.
Fault: strike=140° dip=50°→230° | section tip s=2339 m, apparent dip=-49.8°, throw=140 m (inferred)
Published: {'strike_deg': 140.0, 'dip_deg': 50.0, 'dip_direction': 'SW', 'source': 'Bellahsen; Stanton and Erslev 2004'}
Inferred: {'tip_distance_m': 2339.4220650532666, 'tip_xy': (728867.8714859438, 4944065.0602409635), 'z0_m': 1185.0470345340436, 'throw_m': 139.63056564679664, 'note': "Tip = DEM steep-forelimb toe on A–A'; throw ≈ ½ crest–basin relief for shallow demo"}


Saved: image/cross_section.png
